# Classic Imperative Spark Pipeline (Benchmark & Comparison)

Implements the classic imperative Spark version of the Air Quality pipeline for side-by-side comparison against the declarative Lakeflow pipeline.

In [0]:
import os
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "valeriimatviiv_silver", "Schema")
dbutils.widgets.text("volume", "air_quality_landing", "Volume / Storage Name")
dbutils.widgets.text("custom_landing_path", "", "Custom Landing Path (Optional Override)")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()
custom_landing_path = dbutils.widgets.get("custom_landing_path").strip()

def is_catalog_available(cat_name):
    if not cat_name:
        return False
    try:
        available_catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
        return cat_name in available_catalogs
    except Exception:
        return False

if custom_landing_path:
    base_landing = custom_landing_path.rstrip("/")
    target_prefix = f"{catalog}.{schema}" if is_catalog_available(catalog) else f"{schema}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_prefix}")
elif is_catalog_available(catalog):
    target_prefix = f"{catalog}.{schema}"
    base_landing = f"/Volumes/{catalog}/valeriimatviiv_bronze/{volume}/landing"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_prefix}")
else:
    target_prefix = f"{schema}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    try:
        current_user = spark.sql("SELECT current_user()").collect()[0][0]
        base_landing = f"/Workspace/Users/{current_user}/{volume}/landing"
    except Exception:
        base_landing = f"/Workspace/Shared/{volume}/landing"

checkpoint_base = f"{base_landing}/checkpoints_classic"
stream_landing_path = f"{base_landing}/telemetry_stream"
reference_landing_path = f"{base_landing}/reference"

os.makedirs(f"{checkpoint_base}/bronze_stream", exist_ok=True)
os.makedirs(f"{checkpoint_base}/silver_stream", exist_ok=True)

### Step 1: Classic Bronze Streaming Ingestion (Imperative Auto Loader)

In [0]:
bronze_table_name = f"{target_prefix}.classic_bronze_air_quality"
bronze_checkpoint = f"{checkpoint_base}/bronze_stream"

bronze_query = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{stream_landing_path}/_schema_classic")
    .load(stream_landing_path)
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .trigger(availableNow=True)
    .toTable(bronze_table_name)
)

bronze_query.awaitTermination()
# display(spark.table(bronze_table_name).limit(5))

### Step 2: Classic Bronze Reference Data Ingestion (Batch CSV)

In [0]:
ref_table_name = f"{target_prefix}.classic_bronze_aqi_reference"

df_ref = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(reference_landing_path)
    .withColumn("_reference_loaded_at", F.current_timestamp())
)

df_ref.write.format("delta").mode("overwrite").saveAsTable(ref_table_name)
# display(spark.table(ref_table_name))

### Step 3: Classic Silver Transformation with Manual DQ Filtering & Joins

In [0]:
silver_table_name = f"{target_prefix}.classic_silver_air_quality"
silver_checkpoint = f"{checkpoint_base}/silver_stream"

df_raw = spark.readStream.table(bronze_table_name)
df_ref_table = spark.read.table(ref_table_name)

df_cleaned = (
    df_raw
    .withColumn("recorded_at", F.to_timestamp(F.col("timestamp")))
    .withColumn("pm2_5", F.col("pm2_5").cast("double"))
    .withColumn("pm10", F.col("pm10").cast("double"))
    .withColumn("carbon_monoxide", F.col("carbon_monoxide").cast("double"))
    .withColumn("nitrogen_dioxide", F.col("nitrogen_dioxide").cast("double"))
    .withColumn("sulphur_dioxide", F.col("sulphur_dioxide").cast("double"))
    .withColumn("ozone", F.col("ozone").cast("double"))
    .withColumn("us_aqi", F.col("us_aqi").cast("integer"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
)

# Manual Data Quality Filtering
df_valid = df_cleaned.filter(
    (F.col("event_id").isNotNull()) &
    (F.col("station_id").isNotNull()) &
    (F.col("city").isNotNull()) &
    (F.col("pm2_5").isNull() | (F.col("pm2_5") >= 0.0)) &
    (F.col("pm10").isNull() | (F.col("pm10") >= 0.0)) &
    (F.col("us_aqi").isNull() | ((F.col("us_aqi") >= 0) & (F.col("us_aqi") <= 500))) &
    (F.col("latitude").between(-90.0, 90.0)) &
    (F.col("longitude").between(-180.0, 180.0))
)

df_silver_enriched = (
    df_valid.join(
        df_ref_table,
        (df_valid["us_aqi"] >= df_ref_table["aqi_min"]) & (df_valid["us_aqi"] <= df_ref_table["aqi_max"]),
        how="left"
    )
    .select(
        df_valid["event_id"],
        df_valid["station_id"],
        df_valid["city"],
        df_valid["country"],
        df_valid["latitude"],
        df_valid["longitude"],
        df_valid["recorded_at"],
        df_valid["pm2_5"],
        df_valid["pm10"],
        df_valid["carbon_monoxide"],
        df_valid["nitrogen_dioxide"],
        df_valid["sulphur_dioxide"],
        df_valid["ozone"],
        df_valid["us_aqi"],
        F.coalesce(df_ref_table["category"], F.lit("Unknown")).alias("aqi_category"),
        F.coalesce(df_ref_table["color_code"], F.lit("Gray")).alias("aqi_color_code"),
        df_ref_table["health_implication"],
        df_ref_table["cautionary_statement"],
        df_valid["_ingestion_timestamp"],
        F.current_timestamp().alias("_transformed_timestamp")
    )
)

silver_query = (
    df_silver_enriched
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .toTable(silver_table_name)
)

silver_query.awaitTermination()
# display(spark.table(silver_table_name).limit(10))